# Milan traffic forecasting — final fits and test-week evaluation

Refits every selected model on train **plus** validation and evaluates on the
held-out test week (16–22 Dec 2013) across all three study areas. Writes the
per-area metric tables, the timing table, and the prediction series the Phase 6
figures are drawn from.

### Why all three models run here, not just the LSTM

The timing table is a deliverable. Measuring harmonic ARIMA on an 8-core laptop
and the LSTM on a T4 would produce a table that compares hardware as much as
models. Running all three in one session removes that confound. The `device`
column still separates the LSTM's GPU fits from the two CPU models, because
*that* difference is the finding: on CPU a single LSTM fit took 13.8 h against
2 s for LightGBM.

### What the final fit is allowed to see

Hyperparameters were chosen on validation, so the refit uses train + validation
and the test week stays untouched until prediction. Neither model that stops
early has a held-out set left, so each carries the *complexity* tuning chose:
the LSTM trains for `best_epoch + 1` epochs with patience disabled, LightGBM
uses the `best_iteration` tree count with early stopping off. Both numbers were
fixed before this notebook runs.

The LSTM is run over **3 seeds** and reported as mean ± std, because a single
seed reports one draw and the spread is often comparable to the gap between
models.

### Before running

1. **Accelerator → GPU**. Asserted below rather than silently falling back.
2. **Settings → Internet → On**, for the clone and pip install.
3. No dataset to attach — a clean clone is the whole input.
4. Prefer **Save & Run All (Commit)** so `/kaggle/working` persists.

### What to bring back

`results/tables/` (the metric and timing tables) and `results/predictions/`.

## 1. Settings

In [1]:
# Point this at your own fork/clone before running.
REPO_URL = "https://github.com/Hassan-Adelani-Luqman/milan-traffic-forecasting.git"
BRANCH = "main"

# The test week is the reported result. "stress" (23 Dec - 1 Jan) is the
# held-out holiday period used for failure analysis in Phase 6.
SPLITS = ("test", "stress")

## 2. Clone the repository and install dependencies

In [2]:
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/repo")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
    check=True,
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(REPO_DIR / "requirements-kaggle.txt")],
    check=True,
)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Drop any previously imported src.* modules. Re-running this cell replaces the
# files on disk, but sys.modules still holds the code objects compiled from the
# old ones, so the kernel would keep executing the previous version.
for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[name]
importlib.invalidate_caches()

print("repo:", subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True).stdout.strip())

_check = importlib.import_module("src.final_runs")
assert hasattr(_check, "MODEL_ORDER"), (
    "stale src/ still loaded: restart the kernel "
    "(Run -> Restart & Clear Cell Outputs), then run all cells again"
)
print("src version: OK")

Cloning into '/kaggle/working/repo'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.4/364.4 kB 22.7 MB/s eta 0:00:00
repo: 3a2fd50 Phase 5: LSTM sweep on GPU, all three models selected
src version: OK


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-polars-cu12 26.2.1 requires polars<1.36,>=1.30, but you have polars 1.20.0 which is incompatible.


## 3. Confirm the GPU and record the hardware

In [3]:
import torch

from src.config import load_config
from src.models.lstm import resolve_device
from src.timing import describe_environment, record_hardware

device = resolve_device("auto")
assert device.type == "cuda", (
    f"resolved device is {device!r}, not CUDA. Set Accelerator -> GPU and restart. "
    "On CPU the nine LSTM fits take roughly 31 hours."
)

config = load_config()
config.paths.mkdirs()
assert config.is_kaggle, f"expected the Kaggle overrides, got env={config.env!r}"

env = record_hardware(config.paths.environment_json)
print(describe_environment(env))
print("gpu:", torch.cuda.get_device_name(0))

x86_64 (2 physical / 4 logical cores), 31.3 GB RAM, GPU: Tesla T4 (15 GB), Tesla T4 (15 GB), Linux 6.12.90+, Python 3.12.13
gpu: Tesla T4


## 4. Stage the committed inputs

The series, the area selection and the tuned hyperparameters all live in the
repository. Staging them into the configured paths lets `src.final_runs` run
here exactly as it does locally, with no Kaggle-specific branching in `src/`.

In [4]:
staged = [
    (REPO_DIR / "data/processed/selected_series.parquet",
     config.paths.processed / "selected_series.parquet"),
    (REPO_DIR / "results/tables/selected_areas.json",
     config.paths.tables / "selected_areas.json"),
    (REPO_DIR / "results/tables/selected_hyperparameters.json",
     config.paths.tables / "selected_hyperparameters.json"),
]

for source, destination in staged:
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    print(f"staged  {source.name:<34} -> {destination}")

import json

selected = json.loads(
    (config.paths.tables / "selected_hyperparameters.json").read_text(encoding="utf-8")
)
missing = [m for m in ("harmonic_arima", "lstm", "lightgbm") if m not in selected]
assert not missing, f"no tuned hyperparameters for {missing}; run the tuning stages first"

# The stopping points must have survived tuning, or the refits silently train to
# their ceilings instead of the complexity that was selected.
assert selected["lightgbm"].get("best_iteration", 0) > 0, "LightGBM has no best_iteration"
assert selected["lstm"].get("best_epoch", -1) >= 0, "LSTM has no best_epoch"

print("\ntuned on area", selected["tuning_area"])
print("lstm best_epoch      :", selected["lstm"]["best_epoch"])
print("lightgbm best_iter   :", selected["lightgbm"]["best_iteration"])
print("harmonic order       :", selected["harmonic_arima"]["order"])

staged  selected_series.parquet            -> /kaggle/working/processed/selected_series.parquet
staged  selected_areas.json                -> /kaggle/working/results/tables/selected_areas.json
staged  selected_hyperparameters.json      -> /kaggle/working/results/tables/selected_hyperparameters.json

tuned on area 5161
lstm best_epoch      : 10
lightgbm best_iter   : 320
harmonic order       : [3, 0, 1]


## 5. Final fits on the test week

Three areas x (2 baselines + 3 models), with the LSTM repeated over three
seeds. `walk_forward` with true observed history is the only inference path —
not a recursive rollout — so each forecast uses real observations up to the
step before the one it predicts.

In [5]:
import time

from src.final_runs import run_final

started = time.perf_counter()
results = run_final(config, split_name="test")
print(f"\ntest-week fits: {(time.perf_counter() - started) / 60:.1f} min")


=== square 5161, test split ===
  persistence      MAE 92.80  RMSE 134.88  MAPE 9.19%  MASE 0.267  R2 0.9902
  seasonal_naive   MAE 338.59  RMSE 619.04  MAPE 25.94%  MASE 0.975  R2 0.7934
  harmonic_arima   MAE    83.64  MASE 0.241  R2 0.9911  copy 0.63  (16s train)
  lstm             MAE    92.01 +/- 3.03  MASE 0.265  R2 0.9890  copy 0.71  (10s train)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  lightgbm         MAE    86.59  MASE 0.249  R2 0.9908  copy 0.79  (5s train)

=== square 4159, test split ===
  persistence      MAE 15.95  RMSE 21.54  MAPE 6.98%  MASE 0.195  R2 0.9688
  seasonal_naive   MAE 51.19  RMSE 84.64  MAPE 21.80%  MASE 0.626  R2 0.5179
  harmonic_arima   MAE    13.62  MASE 0.166  R2 0.9766  copy 0.54  (29s train)
  lstm             MAE    21.16 +/- 4.59  MASE 0.259  R2 0.9364  copy 1.14  (9s train)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  lightgbm         MAE    15.21  MASE 0.186  R2 0.9703  copy 0.77  (1s train)

=== square 4556, test split ===
  persistence      MAE 28.86  RMSE 39.62  MAPE 6.60%  MASE 0.257  R2 0.9407
  seasonal_naive   MAE 76.34  RMSE 108.35  MAPE 17.46%  MASE 0.680  R2 0.5568
  harmonic_arima   MAE    25.87  MASE 0.230  R2 0.9545  copy 0.64  (16s train)
  lstm             MAE    28.85 +/- 1.79  MASE 0.257  R2 0.9455  copy 0.80  (9s train)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  lightgbm         MAE    29.93  MASE 0.267  R2 0.9406  copy 0.93  (1s train)

test-week fits: 5.8 min


## 6. The held-out stress split

23 Dec – 1 Jan, never tuned on and never used for selection. The baselines move
in opposite directions here: persistence gets *easier* over the holidays while
seasonal-naive collapses past MASE 1.0, because the weekly pattern it depends on
is exactly what Christmas and New Year destroy. This is where the LightGBM
prediction — that a tree ensemble cannot extrapolate outside its training range
— is tested directly.

In [6]:
started = time.perf_counter()
stress = run_final(config, split_name="stress")
print(f"\nstress-split fits: {(time.perf_counter() - started) / 60:.1f} min")


=== square 5161, stress split ===
  persistence      MAE 68.88  RMSE 107.61  MAPE 12.36%  MASE 0.198  R2 0.9905
  seasonal_naive   MAE 439.88  RMSE 734.57  MAPE 79.19%  MASE 1.266  R2 0.5595
  harmonic_arima   MAE    75.07  MASE 0.216  R2 0.9892  copy 0.74  (15s train)
  lstm             MAE   122.61 +/- 24.04  MASE 0.353  R2 0.9719  copy 1.61  (9s train)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  lightgbm         MAE   116.10  MASE 0.334  R2 0.9734  copy 1.51  (1s train)

=== square 4159, stress split ===
  persistence      MAE 10.12  RMSE 13.30  MAPE 9.56%  MASE 0.124  R2 0.8778
  seasonal_naive   MAE 30.83  RMSE 41.73  MAPE 27.86%  MASE 0.377  R2 -0.2024
  harmonic_arima   MAE     9.93  MASE 0.121  R2 0.8848  copy 0.68  (29s train)
  lstm             MAE    19.81 +/- 1.61  MASE 0.242  R2 0.5996  copy 1.86  (10s train)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  lightgbm         MAE    23.58  MASE 0.288  R2 0.4739  copy 2.23  (1s train)

=== square 4556, stress split ===
  persistence      MAE 23.13  RMSE 31.94  MAPE 8.95%  MASE 0.206  R2 0.9110
  seasonal_naive   MAE 56.60  RMSE 77.43  MAPE 22.48%  MASE 0.504  R2 0.4770
  harmonic_arima   MAE    24.96  MASE 0.222  R2 0.9023  copy 0.86  (17s train)
  lstm             MAE    47.00 +/- 7.18  MASE 0.419  R2 0.7262  copy 1.94  (9s train)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  lightgbm         MAE    55.48  MASE 0.494  R2 0.6299  copy 2.26  (1s train)

stress-split fits: 7.9 min


## 7. Results and timing

In [7]:
import polars as pl

for split in SPLITS:
    path = config.paths.tables / f"final_metrics_all_{split}.csv"
    if not path.exists():
        continue
    print(f"\n===== {split} =====")
    table = pl.read_csv(path).select(
        "square_id", "model", "device", "n_seeds", "mae", "mae_std",
        "mase", "mase_std", "r2", "lag1_copy_ratio", "train_wall_s",
    )
    with pl.Config(tbl_rows=40, tbl_width_chars=160):
        print(table.sort(["square_id", "mase"]))


===== test =====
shape: (15, 11)
┌───────────┬────────────────┬────────┬─────────┬───┬──────────┬─────────┬─────────────────┬──────────────┐
│ square_id ┆ model          ┆ device ┆ n_seeds ┆ … ┆ mase_std ┆ r2      ┆ lag1_copy_ratio ┆ train_wall_s │
│ ---       ┆ ---            ┆ ---    ┆ ---     ┆   ┆ ---      ┆ ---     ┆ ---             ┆ ---          │
│ i64       ┆ str            ┆ str    ┆ i64     ┆   ┆ f64      ┆ f64     ┆ f64             ┆ f64          │
╞═══════════╪════════════════╪════════╪═════════╪═══╪══════════╪═════════╪═════════════════╪══════════════╡
│ 4159      ┆ harmonic_arima ┆ cpu    ┆ 1       ┆ … ┆ 0.0      ┆ 0.97662 ┆ 0.5363          ┆ 29.461       │
│ 4159      ┆ lightgbm       ┆ cpu    ┆ 1       ┆ … ┆ 0.0      ┆ 0.97032 ┆ 0.7658          ┆ 0.765        │
│ 4159      ┆ persistence    ┆ cpu    ┆ 1       ┆ … ┆ 0.0      ┆ 0.96877 ┆ 0.0             ┆ 0.0          │
│ 4159      ┆ lstm           ┆ cuda   ┆ 3       ┆ … ┆ 0.05609  ┆ 0.93637 ┆ 1.138           ┆ 9.249    

In [8]:
timing = pl.read_csv(config.paths.tables / "timing_test.csv")
with pl.Config(tbl_rows=40, tbl_width_chars=160):
    print(timing.sort(["square_id", "train_wall_s"], descending=[False, True]))

shape: (15, 9)
┌────────────────┬───────────┬────────┬─────────┬───┬────────────────┬──────────────────┬───────────────────────┬──────────┐
│ model          ┆ square_id ┆ device ┆ n_seeds ┆ … ┆ train_wall_std ┆ inference_wall_s ┆ inference_ms_per_step ┆ n_params │
│ ---            ┆ ---       ┆ ---    ┆ ---     ┆   ┆ ---            ┆ ---              ┆ ---                   ┆ ---      │
│ str            ┆ i64       ┆ str    ┆ i64     ┆   ┆ f64            ┆ f64              ┆ f64                   ┆ i64      │
╞════════════════╪═══════════╪════════╪═════════╪═══╪════════════════╪══════════════════╪═══════════════════════╪══════════╡
│ harmonic_arima ┆ 4159      ┆ cpu    ┆ 1       ┆ … ┆ 0.0            ┆ 62.7106          ┆ 62.2129               ┆ 22       │
│ lstm           ┆ 4159      ┆ cuda   ┆ 3       ┆ … ┆ 0.087          ┆ 0.0396           ┆ 0.0393                ┆ 202369   │
│ lightgbm       ┆ 4159      ┆ cpu    ┆ 1       ┆ … ┆ 0.0            ┆ 0.0175           ┆ 0.0174              

## 8. Confirm what to download

The metric tables, the timing table and the prediction series. Phase 6 draws
every figure from these, so nothing further needs to be re-run to produce them.

In [9]:
written = sorted(config.paths.tables.glob("final_metrics_*.csv"))
written += sorted(config.paths.tables.glob("timing_*.csv"))
written += sorted(config.paths.predictions.glob("*.parquet"))

for path in written:
    print(f"{path.stat().st_size:>10,}  {path.relative_to(config.paths.results.parent)}")
print(f"\n{len(written)} files to download from the committed version's Output tab.")

     2,169  results/tables/final_metrics_all_stress.csv
     2,112  results/tables/final_metrics_all_test.csv
       843  results/tables/final_metrics_area_4159_stress.csv
       828  results/tables/final_metrics_area_4159_test.csv
       845  results/tables/final_metrics_area_4556_stress.csv
       826  results/tables/final_metrics_area_4556_test.csv
       859  results/tables/final_metrics_area_5161_stress.csv
       836  results/tables/final_metrics_area_5161_test.csv
       846  results/tables/timing_stress.csv
       848  results/tables/timing_test.csv
    57,373  results/predictions/stress_area_4159.parquet
    57,608  results/predictions/stress_area_4556.parquet
    57,797  results/predictions/stress_area_5161.parquet
    41,083  results/predictions/test_area_4159.parquet
    41,001  results/predictions/test_area_4556.parquet
    41,323  results/predictions/test_area_5161.parquet

16 files to download from the committed version's Output tab.
